## DATA PREPROCESSING

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load dataset
df = pd.read_csv("Thales_Group_Manufacturing.csv")


# Combine Date + Timestamp


df['Datetime'] = pd.to_datetime(
    df['Date'] + " " + df['Timestamp']
)

df = df.sort_values("Datetime").reset_index(drop=True)


df.drop(columns=['Date','Timestamp'], inplace=True)


# Encode categorical variable


mode_encoder = LabelEncoder()

df['Operation_Mode'] = mode_encoder.fit_transform(df['Operation_Mode'])


# Target encoding


target_encoder = LabelEncoder()

df['Efficiency_Status'] = target_encoder.fit_transform(
    df['Efficiency_Status']
)


# Scale numerical features


numerical_columns = [
    'Temperature_C',
    'Vibration_Hz',
    'Power_Consumption_kW',
    'Network_Latency_ms',
    'Packet_Loss_%',
    'Quality_Control_Defect_Rate_%',
    'Production_Output_Units',
    'Error_Rate_%'
]

scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])



In [ ]:
# Checking class imbalance

import matplotlib.pyplot as plt

print(df['Efficiency_Status'].value_counts())

df['Efficiency_Status'].value_counts().plot(
    kind='bar',
    color='steelblue'
)

plt.title("Class Distribution")
plt.show()

## FEATURE ENGINEERING

In [ ]:
# Sensor Stability Indicator

df['Sensor_Stability'] = (
    df['Temperature_C'].rolling(10).std().fillna(0)
    +
    df['Vibration_Hz'].rolling(10).std().fillna(0)
)

In [ ]:
# Energy Efficiency Ratio

df['Energy_Efficiency'] = (
    df['Production_Output_Units']
    /
    (df['Power_Consumption_kW'] + 1)
)

In [ ]:
# Error to Output Ratio

df['Error_Output_Ratio'] = (
    df['Error_Rate_%']
    /
    (df['Production_Output_Units'] + 1)
)

In [ ]:
# Network Reliability Score

df['Network_Reliability'] = (
    100
    -
    df['Network_Latency_ms']
    -
    df['Packet_Loss_%']
)

## MODEL DEVELOPMENT

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[
    'Efficiency_Status',
    'Datetime',
    'Machine_ID'
])

y = df['Efficiency_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(
    multi_class='multinomial',
    max_iter=2000
)

lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train,y_train)

pred_rf = rf.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred_rf))

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train,y_train)

pred_xgb = xgb.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred_xgb))

In [ ]:
from sklearn.metrics import classification_report

models = {
    "Logistic": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name, model in models.items():

    prediction = model.predict(X_test)

    print(name)
    print(classification_report(y_test,prediction))

## FEATURE IMPORTANCE

In [ ]:
importance = pd.DataFrame({

    "Feature": X.columns,
    "Importance": rf.feature_importances_

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(20))

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values,X_test)

In [ ]:
shap.force_plot(
    explainer.expected_value[0],
    shap_values[0][0],
    X_test.iloc[0]
)

In [ ]:
probability = rf.predict_proba(X_test)

confidence = probability.max(axis=1)

predictions = rf.predict(X_test)

results = pd.DataFrame({

    "Prediction":predictions,
    "Confidence":confidence

})

print(results.head())

## MODEL DEPLOYMENT

In [ ]:
import streamlit as st
st.sidebar.selectbox(
    "Machine",
    df["Machine_ID"].unique()
)

st.sidebar.selectbox(
    "Operation Mode",
    df["Operation_Mode"].unique()
)

st.sidebar.date_input(
    "Date Range"
)

st.sidebar.slider(
    "Confidence Threshold",
    0.0,
    1.0,
    0.8
)

In [ ]:
st.metric(
    "Predicted Efficiency",
    prediction
)

st.progress(confidence)

st.write(
    f"Confidence: {confidence:.2%}"
)

In [ ]:
fig = px.bar(
    importance.head(10),
    x="Importance",
    y="Feature"
)

st.plotly_chart(fig)

In [ ]:
trend = df.groupby(
    ['Datetime','Machine_ID']
)['Production_Output_Units'].mean().reset_index()

fig = px.line(
    trend,
    x='Datetime',
    y='Production_Output_Units',
    color='Machine_ID'
)

st.plotly_chart(fig)

In [ ]:
fig = px.box(
    df,
    x='Operation_Mode',
    y='Power_Consumption_kW',
    color='Operation_Mode'
)

st.plotly_chart(fig)

In [ ]:
fig = px.scatter(
    df,
    x='Network_Latency_ms',
    y='Production_Output_Units',
    color='Efficiency_Status'
)

st.plotly_chart(fig)